In [16]:
import joblib
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "models").exists():
    ROOT = ROOT.parent

MODEL_DIR = ROOT / "models"

saved = joblib.load(MODEL_DIR / "fraud_model.joblib")
test_data = joblib.load(MODEL_DIR / "test_data.joblib")

model = saved["model"]
threshold = saved["threshold"]

X_test = test_data["X_test"]

print("Selected model:", saved["model_name"])
print("Test transactions:", len(X_test))
print("Threshold:", threshold)

Selected model: XGBoost
Test transactions: 1504
Threshold: 0.12000000000000001


In [17]:
probabilities = model.predict_proba(X_test)[:, 1]

predictions = (probabilities >= threshold).astype(int)

print("Predictions completed.")

Predictions completed.


In [18]:
y_test = test_data["y_test"]

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions,
    "FraudProbability": probabilities
})

results.head(10)

,Actual,Predicted,FraudProbability
0,0,0,0.000402
1,0,0,0.000431
2,0,0,0.000453
3,0,0,0.008085
4,0,0,0.000471
5,0,0,0.000387
6,0,0,0.008273
7,0,0,0.000340
8,0,0,0.001355
9,0,0,0.000492


In [19]:
from sklearn.metrics import (
    precision_score, recall_score,
    f1_score, fbeta_score,
    average_precision_score,
    confusion_matrix
)

print("Precision:", precision_score(y_test, predictions, zero_division=0))
print("Recall:", recall_score(y_test, predictions, zero_division=0))
print("F1 Score:", f1_score(y_test, predictions, zero_division=0))
print("F2 Score:", fbeta_score(y_test, predictions, beta=2, zero_division=0))
print("PR-AUC:", average_precision_score(y_test, probabilities))

print("\nConfusion Matrix:")
pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=[0, 1]),
    index=["Actual Legitimate", "Actual Fraud"],
    columns=["Predicted Legitimate", "Predicted Fraud"]
)

Precision: 0.5
Recall: 0.65
F1 Score: 0.5652173913043478
F2 Score: 0.6132075471698113
PR-AUC: 0.6979382867068836

Confusion Matrix:


,Predicted Legitimate,Predicted Fraud
Actual Legitimate,1471,13
Actual Fraud,7,13


In [22]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data"

transactions = pd.read_csv(DATA / "transactions.csv")
customers = pd.read_csv(DATA / "customers.csv")
answers = pd.read_csv(DATA / "answers.csv")

df = transactions.merge(customers, on="CustomerID")
df = df.merge(answers, on="TransactionID")

review = df.loc[X_test.index, [
    "TransactionID", "AmountINR", "PaymentMethod",
    "TransactionType", "City", "DeviceID",
    "FailedLoginAttempts", "FailedPaymentAttempts",
    "FraudScenario"
]].copy()

review["Actual"] = y_test.values
review["Predicted"] = predictions
review["FraudProbability"] = probabilities

print("Test transactions linked:", len(review))

Test transactions linked: 1504


In [23]:
missed_fraud = review[
    (review["Actual"] == 1) &
    (review["Predicted"] == 0)
]

display(missed_fraud)

,TransactionID,AmountINR,PaymentMethod,TransactionType,City,DeviceID,FailedLoginAttempts,FailedPaymentAttempts,FraudScenario,Actual,Predicted,FraudProbability
8904,TXN006467,150.00,Debit Card,Online Purchase,Kochi,DEV-CUST016-01,1,0,Card testing after failed payment attempts,1,0,0.002392
8905,TXN001408,350.00,Debit Card,Online Purchase,Kochi,DEV-CUST016-01,0,0,Card testing after failed payment attempts,1,0,0.002300
8906,TXN009631,700.00,Credit Card,Online Purchase,Kochi,DEV-CUST016-01,0,2,Card testing after failed payment attempts,1,0,0.001897
8907,TXN005524,1500.00,Credit Card,Online Purchase,Kochi,DEV-CUST016-01,0,0,Card testing after failed payment attempts,1,0,0.001897
8910,TXN004576,12193.81,Debit Card,Online Purchase,Kochi,DEV-CUST016-01,0,0,Card testing after failed payment attempts,1,0,0.028601
9155,TXN006405,10283.30,UPI,Transfer,Mumbai,DEV-RAPID-05,1,0,Rapid transfers to multiple beneficiaries,1,0,0.002295
9159,TXN003522,13334.12,UPI,Transfer,Mumbai,DEV-RAPID-05,0,0,Rapid transfers to multiple beneficiaries,1,0,0.029969


In [24]:
false_alerts = review[
    (review["Actual"] == 0) &
    (review["Predicted"] == 1)
]

display(false_alerts)

,TransactionID,AmountINR,PaymentMethod,TransactionType,City,DeviceID,FailedLoginAttempts,FailedPaymentAttempts,FraudScenario,Actual,Predicted,FraudProbability
8516,TXN001596,19831.20,Debit Card,Transfer,Hyderabad,DEV-CUST050-01,1,0,Legitimate,0,1,0.877219
8681,TXN001670,1767.48,UPI,Cash Withdrawal,Kochi,DEV-NEW-813007,0,0,Legitimate,0,1,0.334680
8808,TXN004216,22252.49,IMPS,Online Purchase,Ahmedabad,DEV-NEW-592546,0,0,Legitimate,0,1,0.840493
8990,TXN001621,16709.70,NEFT,Transfer,Kolkata,DEV-CUST008-01,0,0,Legitimate,0,1,0.328362
9084,TXN009264,12496.28,NEFT,Transfer,Delhi,DEV-CUST052-01,0,0,Legitimate,0,1,0.203465
9188,TXN002124,26897.10,IMPS,Transfer,Bengaluru,DEV-CUST092-02,0,0,Legitimate,0,1,0.164083
9326,TXN004961,22486.40,IMPS,Transfer,Bengaluru,DEV-NEW-644932,0,0,Legitimate,0,1,0.953124
9442,TXN002054,16200.29,NEFT,Online Purchase,Kolkata,DEV-NEW-440311,0,1,Legitimate,0,1,0.344196
9613,TXN007695,47481.80,Debit Card,Transfer,Kolkata,DEV-CUST002-02,0,1,Legitimate,0,1,0.614744
9805,TXN004233,19083.70,UPI,Transfer,Delhi,DEV-CUST100-01,0,0,Legitimate,0,1,0.975814


In [25]:
print("Missed fraud by scenario:")
display(missed_fraud["FraudScenario"].value_counts())

print("False alerts by payment method:")
display(false_alerts["PaymentMethod"].value_counts())

Missed fraud by scenario:


FraudScenario
Card testing after failed payment attempts    5
Rapid transfers to multiple beneficiaries     2
Name: count, dtype: int64

False alerts by payment method:


PaymentMethod
IMPS          4
NEFT          3
UPI           3
Debit Card    2
RTGS          1
Name: count, dtype: int64